## Local Inference on GPU 
Model page: https://huggingface.co/grandhigh/Chatterbox-TTS-Indonesian

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/grandhigh/Chatterbox-TTS-Indonesian)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
!pip uninstall -y torchvision
!pip install "setuptools<70.0.0" "numpy<2.0" --no-deps --force-reinstall
!pip install chatterbox-tts --no-deps
!pip install perth librosa transformers huggingface_hub pykakasi spacy peft

In [1]:
import os
import sys
import torch
import perth
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# Patch Perth watermarker jika dummy
if not hasattr(perth, "PerthImplicitWatermarker") or perth.PerthImplicitWatermarker is None:
    class DummyWatermarker:
        def __init__(self, *args, **kwargs): pass
        def apply_watermark(self, wav, *args, **kwargs): return wav
    perth.PerthImplicitWatermarker = DummyWatermarker

import torchaudio as ta
from chatterbox.tts import ChatterboxTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan device: {device}")

# 1. Load base model ChatterboxTTS
print("Memuat base model...")
model = ChatterboxTTS.from_pretrained(device=device)

# 2. Unduh dan timpa weights dengan checkpoint 'grandhigh/Chatterbox-TTS-Indonesian'
print("Mengunduh weights bahasa Indonesia dari grandhigh/Chatterbox-TTS-Indonesian...")
MODEL_REPO = "grandhigh/Chatterbox-TTS-Indonesian"
CHECKPOINT_FILENAME = "t3_cfg.safetensors"

t3_path = hf_hub_download(repo_id=MODEL_REPO, filename=CHECKPOINT_FILENAME)

# Load weights T3 khusus Indonesia
t3_state_dict = load_file(t3_path)
model.t3.load_state_dict(t3_state_dict)
print("✅ Weights Bahasa Indonesia berhasil dimasukkan ke model!")

Menggunakan device: cuda
Memuat base model...


/usr/local/lib/python3.12/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


Mengunduh weights bahasa Indonesia dari grandhigh/Chatterbox-TTS-Indonesian...
✅ Weights Bahasa Indonesia berhasil dimasukkan ke model!


In [9]:
import os
import glob
import re
import torch
import torchaudio as ta

# 1. Teks Panjang Anda
full_text = """
Mencari pekerjaan di Indonesia hari ini sering kali terasa seperti memecahkan teka-teki tanpa jawaban. Coba buka platform pencarian kerja mana pun—LinkedIn, JobStreet, atau sekadar memantau lini masa media sosial. Dalam hitungan detik, gumpalan kualifikasi yang saling bertabrakan secara logika akan langsung terlihat.
Sebuah posisi berlabel entry-level—posisi yang secara definisi dirancang untuk lulusan baru atau individu tanpa rekam jejak industri—secara eksplisit menuntut pengalaman relevan selama dua hingga tiga tahun. Di saat bersamaan, batas usia maksimal dipatok sangat rendah: 21 atau 22 tahun. Secara kalkulasi matematis, kualifikasi ini jelas tidak masuk akal bagi seorang lulusan perguruan tinggi yang menempuh jalur pendidikan reguler.
Fenomena ini makin runyam saat melihat deskripsi pekerjaan. Satu posisi sering kali dibebani tugas dari tiga hingga empat divisi berbeda. Posisi Social Media Specialist, misalnya, diwajibkan menyusun strategi konten, menulis skrip, mengambil gambar, menyunting video, membuat desain grafis, hingga mengelola anggaran iklan digital. Semua kerumitan peran ini kemudian ditutup dengan satu muara: tawaran kompensasi finansial yang persis menyentuh garis batas minimum provinsi.
Di ruang publik, keanehan ini kerap disederhanakan sebagai bentuk keserakahan pemilik modal atau ketidakmampuan praktisi HRD dalam merancang standar kualifikasi. Tudingan tersebut tentu tidak sepenuhnya salah. Kendati demikian, reaksi emosional semacam itu gagal menyentuh akar masalah yang sebenarnya.
Jika dibedah menggunakan instrumen ekonomi tenaga kerja dan sosiologi industri, anomali syarat lowongan kerja di Indonesia bukanlah sekadar kecerobohan administrasi, melainkan manifestasi dari krisis struktural yang jauh lebih mendalam.
"""

# 2. Pembersihan & Pemotongan Teks per Kalimat
# Menghapus simbol khusus yang bisa membingungkan tokenizer (seperti '—')
cleaned_text = full_text.replace("—", ", ").strip()
# Split berdasarkan tanda titik, tanda tanya, atau titik dua
sentences = [s.strip() for s in re.split(r'(?<=[.?!:])\s+', cleaned_text) if s.strip()]

# 3. Ambil Audio Prompt
search_path = "/kaggle/input/**/*.wav"
found_files = glob.glob(search_path, recursive=True)
AUDIO_PROMPT_PATH = found_files[0] if found_files else None

if AUDIO_PROMPT_PATH and os.path.exists(AUDIO_PROMPT_PATH):
    print(f"✅ Menggunakan audio prompt: {AUDIO_PROMPT_PATH}")
    
    # Trim audio acuan ke 8 detik pertama untuk konsistensi timbre
    info = ta.info(AUDIO_PROMPT_PATH)
    waveform, sr = ta.load(AUDIO_PROMPT_PATH, frame_offset=0, num_frames=int(info.sample_rate * 8))
    TEMP_PROMPT = "trimmed_prompt.wav"
    ta.save(TEMP_PROMPT, waveform, sr)

    print(f"🔄 Memproses total {len(sentences)} kalimat...")
    
    generated_chunks = []
    
    # 4. Generasi Audio per Kalimat
    for idx, sentence in enumerate(sentences, 1):
        print(f"[{idx}/{len(sentences)}] Generasi: '{sentence[:40]}...'")
        
        wav_chunk = model.generate(
            sentence,
            audio_prompt_path=TEMP_PROMPT,
            temperature=0.25,        # Nilai rendah menjaga stabilitas aksen Indonesia
            repetition_penalty=1.2
        )
        
        # Tambahkan sedikit jeda hening (0.3 detik) antar kalimat agar terdengar alami
        silence_duration = int(model.sr * 0.3)
        silence = torch.zeros((wav_chunk.shape[0], silence_duration))
        
        generated_chunks.append(wav_chunk)
        generated_chunks.append(silence)

    # 5. Gabungkan Seluruh Audio Chunk
    print("\n🔗 Menggabungkan seluruh potongan audio...")
    full_audio = torch.cat(generated_chunks, dim=1)
    
    # Simpan ke file akhir
    output_filename = "naskah_panjang_indo.wav"
    ta.save(output_filename, full_audio, model.sr)
    print(f"🎉 SELESAI! File audio lengkap berhasil disimpan sebagai: '{output_filename}'")

else:
    print("❌ File sampel suara tidak ditemukan di /kaggle/input/.")

✅ Menggunakan audio prompt: /kaggle/input/datasets/eruru07/andre-data/ANDREFINAL22 AFTER EDIT.wav
🔄 Memproses total 16 kalimat...
[1/16] Generasi: 'Mencari pekerjaan di Indonesia hari ini ...'


Sampling:  18%|█▊        | 182/1000 [00:05<00:22, 36.23it/s]


[2/16] Generasi: 'Coba buka platform pencarian kerja mana ...'


Sampling:  18%|█▊        | 184/1000 [00:05<00:22, 35.84it/s]


[3/16] Generasi: 'Dalam hitungan detik, gumpalan kualifika...'


Sampling:  18%|█▊        | 180/1000 [00:04<00:22, 36.05it/s]


... (23 outputs left out)

Sampling:  19%|█▉        | 188/1000 [00:05<00:22, 36.08it/s]


[16/16] Generasi: 'Jika dibedah menggunakan instrumen ekono...'


Sampling:  36%|███▌      | 357/1000 [00:10<00:18, 35.64it/s]



🔗 Menggabungkan seluruh potongan audio...
🎉 SELESAI! File audio lengkap berhasil disimpan sebagai: 'naskah_panjang_indo.wav'


In [10]:
import IPython.display as ipd

print("🔊 Audio 1 (Standar):")
ipd.display(ipd.Audio("test-1.wav"))

print("🔊 Audio 2 (Voice Cloning):")
ipd.display(ipd.Audio("naskah_panjang_indo.wav"))

🔊 Audio 1 (Standar):


🔊 Audio 2 (Voice Cloning):
